## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, count, countDistinct, avg, sum as _sum, max as _max, min as _min, when, coalesce, row_number, year, month, to_date

## Read all silver tables

In [0]:
df_orders = spark.table("olist.silver.orders")
df_items = spark.table("olist.silver.order_items")
df_products = spark.table("olist.silver.products")
df_customers = spark.table("olist.silver.customers")
df_sellers = spark.table("olist.silver.sellers")
df_payments = spark.table("olist.silver.order_payments")
df_reviews = spark.table("olist.silver.order_reviews")
df_translation = spark.table("olist.silver.product_category_name_translation")

## Add English category names to products table

In [0]:
%sql
-- Show products with their English category name.
-- COALESCE keeps the original name when there is no translation row.
SELECT
    p.*,
    COALESCE(t.product_category_name_english, p.product_category_name) AS product_category_name_english
FROM olist.silver.products p
LEFT JOIN olist.silver.product_category_name_translation t
       ON p.product_category_name = t.product_category_name

## Aggregate payments per order 

In [0]:
%sql
-- One row per order with review summary.
SELECT
    order_id,
    AVG(review_score)         AS avg_review_score,
    MIN(review_score)         AS min_review_score,
    MAX(review_score)         AS max_review_score,
    COUNT(review_id)          AS review_count,
    MAX(review_creation_date) AS last_review_date
FROM olist.silver.order_reviews
GROUP BY order_id


##  Build the enriched table

In [0]:
%sql
SELECT *
FROM olist.silver.order_items oi
INNER JOIN olist.silver.orders o ON oi.order_id = o.order_id
LEFT JOIN olist.silver.products p ON oi.product_id = p.product_id
LEFT JOIN olist.silver.customers c ON o.customer_id = c.customer_id
LEFT JOIN olist.silver.sellers s on s.seller_id = oi.seller_id
LEFT JOIN olist.silver.order_payments py ON o.order_id = py.order_id
LEFT JOIN olist.silver.order_reviews r ON o.order_id = r.order_id